# Appendix 2: Large-scale GVCF - the 5-step procedure
*INRAe-IFB-INSERM Bioinformatics School 2026*

* Maria BERNARD - INRAE

This appendix complements section **III.3.2 GATK with GVCF output** of the main practical session. It details the **5-step** procedure recommended by the Broad Institute whenever you work with a **large number of samples** and/or a **large genome**, replacing the 3-step procedure (`HaplotypeCaller` → `CombineGVCFs` → `GenotypeGVCFs`) seen in the practical session.

source:
* [GATK - GenomicsDBImport](https://gatk.broadinstitute.org/hc/en-us/articles/30332006200603-GenomicsDBImport)
* [GATK - The logic of joint calling for germline short variants](https://gatk.broadinstitute.org/hc/en-us/articles/360035890431-The-logic-of-joint-calling-for-germline-short-variants)


## I - Why replace `CombineGVCFs` with `GenomicsDBImport`?

`CombineGVCFs` (3-step procedure of the main practical session) merges all the individual GVCFs into **a single** multi-sample GVCF file, re-reading and rewriting all the data at each call.

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; color: #856404; padding: 10px; border-radius: 4px;">
    ⚠️Warning<br>
    This approach does not scale: on a large genome or with many samples, the combined GVCF file becomes huge, computing time explodes, and adding a single new sample forces you to merge everything again.
</div>

[GenomicsDBImport](https://gatk.broadinstitute.org/hc/en-us/articles/30332006200603-GenomicsDBImport) solves this problem by importing the individual GVCFs into a **genomic database**, organised by genome interval rather than as a flat file. This makes it possible to:
* **parallelise** genotyping over sub-regions of the genome (several simultaneous jobs, one per interval);
* **incrementally add** new samples to an existing database, without redoing everything.

<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px;">
    ℹ️ <strong>Note</strong><br>
    Reminder from the main practical session: the Broad Institute recommends switching to this 5-step procedure as soon as you exceed a few dozen samples or work on a large genome.
</div>


## II - Pipeline overview

```
 sample1.g.vcf    sample2.g.vcf    ...    sampleN.g.vcf      (HaplotypeCaller -ERC GVCF, one per sample)
        │                 │                      │
        │                 │                      │
        │                 │                      │
        │ ┌────────────────────────────────────┐ │
        │ │ SplitIntervals (whole genome)      │ │
        │ │  →  intervals I1, I2, ... , Ik     │ │
        │ └────────────────────────────────────┘ │
        ▼               ▼ │                      ▼
        ┌─────────────────┼──────────────────────┐
        ▼                 ▼                      ▼
  GenomicsDBImport   GenomicsDBImport     GenomicsDBImport      (one per interval Ii, all samples)
     (I1)                (I2)                  (Ik)
        │                 │                     │
        ▼                 ▼                     ▼
  GenotypeGVCFs      GenotypeGVCFs         GenotypeGVCFs         (one per interval Ii)
     (I1)                (I2)                  (Ik)
        │                 │                     │
        └─────────────────┼─────────────────────┘
                          ▼
                     GatherVcfs
                          │
                          ▼
          cohort_genome.vcf.gz   (multi-sample VCF, whole genome)
```

In order:
1. **HaplotypeCaller** in GVCF mode, one per sample (already seen in the main practical session, unchanged)
2. **SplitIntervals**: splits the genome into balanced intervals, to parallelise the following steps
3. **GenomicsDBImport**, run on each interval: imports all the individual GVCFs into a GenomicsDB workspace specific to that interval
4. **GenotypeGVCFs**, run on each GenomicsDB workspace (one per interval): produces one multi-sample VCF per interval
5. **GatherVcfs**: concatenates the VCFs obtained for each interval to produce a single multi-sample VCF over the whole genome


## III - Step by step

In [ ]:
module load gatk4/4.6.1.0
mkdir -p Results/Variant_calling/large_cohort


### 1. HaplotypeCaller in GVCF mode (reminder)

This step is strictly identical to the one in the main practical session (part III.3.2.a): one GVCF per sample. The only difference here, since we are in the "large genome" case: we no longer restrict the analysis to a small interval with `--intervals`, we run on the whole reference genome.


In [ ]:
mkdir -p Results/Variant_calling/large_cohort/gvcf

for sample in milca_40995 milca_40998 control_41011 control_41012; do
  gatk HaplotypeCaller --java-options '-Xmx8G' \
     --input Results/Variant_calling/BaseRecalibration/${sample}_recal.bam \
     --reference data/genome/Bos_taurus_GCF_002263795.3_ARS-UCD2.0_chr2_chr6.fasta \
     --base-quality-score-threshold 18 \
     --min-base-quality-score 10 \
     --minimum-mapping-quality 20 \
     --sample-ploidy 2 \
     --emit-ref-confidence GVCF \
     --native-pair-hmm-threads 4 \
     --output Results/Variant_calling/large_cohort/gvcf/${sample}.g.vcf
done


### 2. SplitIntervals: splitting the genome into balanced intervals

[SplitIntervals](https://gatk.broadinstitute.org/hc/en-us/articles/30332044447003-SplitIntervals) splits the genome (or a list of intervals) into `N` sub-intervals of comparable size (in number of bases), to parallelise the following steps.

Main options:
* `--scatter-count`: number of interval files to produce (to be adapted to the number of available cores/jobs, see part IV)
* `--subdivision-mode`: `BALANCING_WITHOUT_INTERVAL_SUBDIVISION` keeps the original intervals (useful if you already have a chromosome file) without cutting them, unlike the default `INTERVAL_SUBDIVISION` mode which can split a chromosome into several pieces
* `--output`: output directory (one `.interval_list` file per interval produced)


In [ ]:
mkdir -p Results/Variant_calling/large_cohort/intervals

gatk SplitIntervals \
   --reference data/genome/Bos_taurus_GCF_002263795.3_ARS-UCD2.0_chr2_chr6.fasta \
   --scatter-count 4 \
   --subdivision-mode BALANCING_WITHOUT_INTERVAL_SUBDIVISION \
   --output Results/Variant_calling/large_cohort/intervals


<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px;">
    ℹ️ <strong>Note</strong><br>
    On our dataset reduced to 2 chromosomes (<code>chr2</code>/<code>chr6</code>), 4 intervals are enough to illustrate the principle. On a complete bovine genome (29 autosomes + X), you would typically use several dozen to a few hundred intervals.
</div>


### 3. GenomicsDBImport: importing the GVCFs, one workspace per interval

[GenomicsDBImport](https://gatk.broadinstitute.org/hc/en-us/articles/30332006200603-GenomicsDBImport) imports the GVCFs of **all** samples into a GenomicsDB database specific to a given interval.

Rather than repeating `--variant` for each sample, we use a `--sample-name-map` file: a tab-delimited text file `sample_name <TAB> gvcf_path`, one line per sample.

Main options:
* `--genomicsdb-workspace-path`: path of the workspace to create (must be a **non-existent** directory)
* `--sample-name-map`: sample → GVCF mapping file
* `--intervals`: the interval to process (one of the files produced by `SplitIntervals`)
* `--batch-size`: number of samples opened simultaneously (0 = all at once; to be specified, then reduced if you run out of memory on large cohorts)

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; color: #856404; padding: 10px; border-radius: 4px;">
    ⚠️Warning<br>
    The temporary disk space used by <code>GenomicsDBImport</code> may exceed the space available in <code>/tmp</code> by default, especially with many intervals or samples: remember to set <code>--tmp-dir</code> to a directory with enough space.
</div>


In [ ]:
# Sample -> GVCF mapping file
for gvcf in Results/Variant_calling/large_cohort/gvcf/*.g.vcf; do
   sample=$(basename ${gvcf} | sed -E 's/\.g\.vcf$//')
   echo -e "${sample}\t${gvcf}" >> Results/Variant_calling/large_cohort/sample_map.tsv
done


In [ ]:
mkdir -p Results/Variant_calling/large_cohort/genomicsdb Results/Variant_calling/large_cohort/tmp

for interval in Results/Variant_calling/large_cohort/intervals/*-scattered.interval_list; do
   name=$(basename ${interval} .interval_list)
   gatk GenomicsDBImport --java-options '-Xmx8G' \
      --sample-name-map Results/Variant_calling/large_cohort/sample_map.tsv \
      --genomicsdb-workspace-path Results/Variant_calling/large_cohort/genomicsdb/${name} \
      --intervals ${interval} \
      --batch-size 50 \
      --reader-threads 4 \
      --tmp-dir Results/Variant_calling/large_cohort/tmp
done


### 4. GenotypeGVCFs: genotyping each GenomicsDB workspace

[GenotypeGVCFs](https://gatk.broadinstitute.org/hc/en-us/articles/30332044749979-GenotypeGVCFs) (used in the main practical session, see part 3.2.c) is then run once per interval, pointing directly to the corresponding GenomicsDB workspace via the `gendb://` URI.


In [ ]:
mkdir -p Results/Variant_calling/large_cohort/vcf_by_interval

for interval in Results/Variant_calling/large_cohort/intervals/*-scattered.interval_list; do
   name=$(basename ${interval} .interval_list)
   gatk GenotypeGVCFs --java-options '-Xmx8G' \
      --reference data/genome/Bos_taurus_GCF_002263795.3_ARS-UCD2.0_chr2_chr6.fasta \
      --variant gendb://Results/Variant_calling/large_cohort/genomicsdb/${name} \
      -stand-call-conf 30 \
      --tmp-dir Results/Variant_calling/large_cohort/tmp \
      --output Results/Variant_calling/large_cohort/vcf_by_interval/${name}.vcf.gz
done


### 5. GatherVcfs: rebuilding the whole-genome VCF

[GatherVcfs](https://gatk.broadinstitute.org/hc/en-us/articles/30332005382555-GatherVcfs-Picard) concatenates the per-interval VCFs into a single VCF, provided they are given **in genomic order** and all share the same set of samples.

<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px;">
    ℹ️ <strong>Note</strong><br>
    The files <code>0000-scattered.vcf.gz</code>, <code>0001-scattered.vcf.gz</code>, ... produced above are already in genomic order (<code>SplitIntervals</code> numbers them following the order of the reference <code>.dict</code>): no need to sort them.
</div>


In [ ]:
INPUTS=""
for vcf in Results/Variant_calling/large_cohort/vcf_by_interval/*.vcf.gz; do
   INPUTS="${INPUTS} --INPUT ${vcf}"
done

gatk GatherVcfs ${INPUTS} --java-options '-Xmx8G' \
   --OUTPUT Results/Variant_calling/large_cohort/4samples_genome.vcf.gz


## IV - How many intervals should you choose (`--scatter-count`)?

There is no single value: the right trade-off depends on several factors.

* **Too few intervals**: little parallelisation, you do not take advantage of the available cores/jobs.
* **Too many intervals**: each `GenomicsDBImport`/`GenotypeGVCFs` has a fixed start-up cost (opening the GVCFs, writing metadata); multiplying small intervals can waste more time in overhead than is gained through parallelisation, and multiplies the number of small files to manage.

<div style="background-color: #dafbe1; border-left: 6px solid #1f883d; padding: 10px;">
    💡 <strong>Tip</strong><br>
    A good starting point is to aim for a number of intervals close to the number of jobs you can submit in parallel on your cluster (or a small multiple of it), while keeping each interval large enough to absorb the start-up cost (typically several tens of Mb per interval on a mammalian genome).
</div>


## V - Adding new samples without redoing everything

One of the main advantages of `GenomicsDBImport` over `CombineGVCFs`: an existing GenomicsDB workspace can be updated incrementally with the `--genomicsdb-update-workspace-path` option (instead of `--genomicsdb-workspace-path`), to add the GVCFs of new samples without re-importing the samples already present.

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; color: #856404; padding: 10px; border-radius: 4px;">
    ⚠️Warning<br>
    An incremental update that fails midway can leave the workspace in an inconsistent state. Always back up your GenomicsDB workspace before adding new samples to it.
</div>


## VI - References

* [GATK - SplitIntervals](https://gatk.broadinstitute.org/hc/en-us/articles/30332044447003-SplitIntervals)
* [GATK - GenomicsDBImport](https://gatk.broadinstitute.org/hc/en-us/articles/30332006200603-GenomicsDBImport)
* [GATK - GenotypeGVCFs](https://gatk.broadinstitute.org/hc/en-us/articles/30332044749979-GenotypeGVCFs)
* [GATK - GatherVcfs (Picard)](https://gatk.broadinstitute.org/hc/en-us/articles/30332005382555-GatherVcfs-Picard)
* [GATK - The logic of joint calling for germline short variants](https://gatk.broadinstitute.org/hc/en-us/articles/360035890431-The-logic-of-joint-calling-for-germline-short-variants)
* [GATK - HaplotypeCaller](https://gatk.broadinstitute.org/hc/en-us/articles/30332006386459-HaplotypeCaller)
* [GATK - CombineGVCFs (3-step procedure, main practical session)](https://gatk.broadinstitute.org/hc/en-us/articles/30332058987163-CombineGVCFs)
